# 🎙️ [시니어사연 보이스 무료제작기] 구글 코랩 무료 T4 GPU 고속 워커

**AI 1인 기업 대표님을 위한 100% 무료 클라우드 GPU 오디오 가속 서버입니다.**

### ⚡ 사용 방법 (단 2단계!)
1. 상단 메뉴에서 **[런타임] → [모두 실행]** (또는 `Ctrl + F9`)을 누릅니다.
2. 맨 마지막 셀의 실행 결과에 나오는 **`https://xxxx.trycloudflare.com`** 주소를 복사하여 로컬 웹앱의 **[코랩 워커 URL]**에 붙여넣으면 끝입니다!

> 💡 **하드웨어 확인:** 구글 코랩 메뉴 [런타임] → [런타임 유형 변경]에서 **T4 GPU**가 선택되어 있는지 확인해 주세요.

In [ ]:
# 1. GPU 하드웨어 확인 및 필수 환경 체크
!nvidia-smi

In [ ]:
# 2. 고음질 음성 생성 패키지 및 서버 라이브러리 초고속 설치
!pip install -q fastapi uvicorn pydantic edge-tts pydub soundfile nest-asyncio

In [ ]:
# 3. 토큰 필요 없는 무료 Cloudflare Tunnel 클라이언트 설치
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✓ Cloudflared 터널 엔진 준비 완료!")

In [ ]:
# 4. 남녀 연령대별 캐릭터 보이스 매핑 및 고품질 FastAPI 워커 서버 작성
import asyncio
import io
import os
import tempfile
import edge_tts
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading
import nest_asyncio
nest_asyncio.apply()

app = FastAPI(title="시니어사연 보이스 무료제작기 Worker")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 한국어 고품질 신경망 보이스 및 피치/속도 파라미터 매핑
# (F5-TTS 레퍼런스 스타일 및 고품질 뉴럴 보이스 조율)
VOICE_CONFIG = {
    # 1. 메인 내레이션: 여성 40~50대 중년, 중후하고 편안한 라디오/오디오북 톤
    "NARRATION": {
        "voice": "ko-KR-SunHiNeural",
        "rate": "-4%",
        "pitch": "-3Hz"
    },
    # 2. 주인공 이정희: 65세 여성, 삶의 연륜과 온화함, 단호함
    "FEMALE_SENIOR_LEAD": {
        "voice": "ko-KR-SunHiNeural",
        "rate": "-6%",
        "pitch": "-6Hz"
    },
    # 3. 남편 김병수: 70세 남성 악역, 거칠고 쉰 목소리, 버럭 호통 톤
    "MALE_SENIOR_VILLAIN": {
        "voice": "ko-KR-InJoonNeural",
        "rate": "+2%",
        "pitch": "-8Hz"
    },
    # 4. 아들 김태호: 40세 남성 중년, 주눅들고 우유부단한 톤, 후회
    "MALE_MIDDLE_SON": {
        "voice": "ko-KR-InJoonNeural",
        "rate": "-2%",
        "pitch": "+0Hz"
    },
    # 5. 며느리 최윤지: 38세 여성 청년/중년초반, 맑고 또렷하며 강단 있는 톤
    "FEMALE_YOUNG_HELPER": {
        "voice": "ko-KR-SunHiNeural",
        "rate": "+0%",
        "pitch": "+4Hz"
    },
    # 6. 판사: 50대 남성, 근엄하고 신뢰감 넘치는 톤
    "MALE_MIDDLE_JUDGE": {
        "voice": "ko-KR-InJoonNeural",
        "rate": "-5%",
        "pitch": "-4Hz"
    },
    # 7. 변호사: 30대 남성 전문직 차분한 톤
    "MALE_YOUNG_LAWYER": {
        "voice": "ko-KR-InJoonNeural",
        "rate": "+1%",
        "pitch": "+2Hz"
    },
    # 8. 어르신/노인 손님들
    "MALE_SENIOR_EXTRA": {
        "voice": "ko-KR-InJoonNeural",
        "rate": "-8%",
        "pitch": "-6Hz"
    },
    "FEMALE_SENIOR_EXTRA": {
        "voice": "ko-KR-SunHiNeural",
        "rate": "-6%",
        "pitch": "-4Hz"
    },
    # 9. 청년 봉사자/기자/경찰
    "FEMALE_YOUNG_VOLUNTEER": {
        "voice": "ko-KR-SunHiNeural",
        "rate": "+2%",
        "pitch": "+6Hz"
    },
    "MALE_YOUNG_OFFICER": {
        "voice": "ko-KR-InJoonNeural",
        "rate": "+2%",
        "pitch": "-1Hz"
    }
}

class TTSRequest(BaseModel):
    speaker_id: str = "NARRATION"
    text: str
    rate_override: str = None
    pitch_override: str = None

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "app": "시니어사연 보이스 무료제작기 Worker",
        "available_voices": list(VOICE_CONFIG.keys())
    }

@app.post("/generate")
async def generate_speech(req: TTSRequest):
    cfg = VOICE_CONFIG.get(req.speaker_id, VOICE_CONFIG["NARRATION"])
    voice = cfg["voice"]
    rate = req.rate_override or cfg["rate"]
    pitch = req.pitch_override or cfg["pitch"]
    
    communicate = edge_tts.Communicate(req.text, voice, rate=rate, pitch=pitch)
    audio_stream = io.BytesIO()
    
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            audio_stream.write(chunk["data"])
            
    audio_bytes = audio_stream.getvalue()
    if not audio_bytes:
        raise HTTPException(status_code=500, detail="음성 생성 실패")
        
    return Response(content=audio_bytes, media_type="audio/mpeg")

print("✓ FastAPI 워커 엔드포인트 정의 완료!")

In [ ]:
# 5. 백그라운드 서버 구동 및 무료 Cloudflare 터널 URL 생성
import subprocess
import time
import re

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Cloudflare Tunnel 실행 및 주소 추출
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

print("=" * 65)
print("🎉 [시니어사연 보이스 무료제작기] 코랩 워커가 활성화되었습니다!")
print(f"👉 아래의 터널 URL을 복사하여 로컬 웹앱에 붙여넣으세요:")
print(f"\n🔗 {tunnel_url}\n")
print("=" * 65)